# solveit-tex

A utility for creating LaTeX papers using SolveIT, to facilitate Human-Authored, Computer-Interrogated (HACI) writing workflows.

The idea is to be able to work in Solvit and see somewhat of what you're working on and be able to interrogate it via the AI incrementally and then have it generate a paper.

Full interoperability between Solve It and LaTeX isn't entirely something I'm going for, but I'm trying to make it so that things that render In one medium, render in the other to as high a degree as possible.

In [ ]:
#| export
import os, subprocess, json, re, sys
from pathlib import Path
from IPython.display import HTML, display

## Utilities called by other routines

In [ ]:
#| export
async def get_curr_dialog_path():
    from dialoghelper.solveitskill import curr_dialog, realpath
    name = (await curr_dialog())['name']
    return f'{await realpath("/")}/{name}.ipynb'

In [ ]:
curr_dialog_path = await get_curr_dialog_path()
print(curr_dialog_path)
print(type(curr_dialog_path))

/app/data/solveit-tex/solveit-tex.ipynb


<class 'str'>


In [ ]:
#| export
def get_private_url(path: str):
    "Get the private URL for a file on the solveit cloud instance"
    server = os.getenv('PRIVATE_DOMAIN')
    if not server: raise ValueError("PRIVATE_DOMAIN not set")
    path = os.path.abspath(path)
    return f"https://{server}.solve.it.com{path.replace('/app/data', '/static')}"

In [ ]:
get_private_url(curr_dialog_path) 

'https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/solveit-tex.ipynb'

### Figures

In [ ]:
#| export
def parse_figure(line: str):
    r"""Parse markdown figure syntax with multiple images on one line: ![alt1](img1.png) ![alt2](img2.png)\{width=45% #fig:label}
     Images on one line get grouped into a single figure, with the final caption and label being the one used for the group"""
    import re
    
    # Look for escaped attributes at the end: \{...}
    attrs = ''
    attr_match = re.search(r'\\\{([^}]*)\}\s*$', line)
    if attr_match:
        attrs = attr_match.group(1)
        line = line[:attr_match.start()]  # Remove the attributes part
    
    # Find all image patterns on the line
    pattern = r'!\[([^\]]*)\]\(([^)]+)\)'
    matches = re.findall(pattern, line.strip())
    
    if not matches: return None
    
    images = []
    caption = ""
    
    for i, (alt, path) in enumerate(matches):
        img = {'path': path.strip()}
        images.append(img)
        
        # Last image sets the caption and label
        if i == len(matches) - 1:
            caption = alt
    
    # Extract width and label from attributes
    width_m = re.search(r'width=([^\s#]+)', attrs)
    label_m = re.search(r'#fig:([^\s}]+)', attrs)
    width = width_m.group(1) if width_m else None
    label = label_m.group(1) if label_m else None
    
    # Apply width to all images if specified
    if width:
        for img in images:
            img['width'] = width
    
    # If caption is just a filename, treat as no caption
    if caption:
        path_basenames = [Path(p['path']).name for p in images]
        if caption in path_basenames or caption in [p['path'] for p in images]:
            caption = ""
    
    return {'caption': caption, 'images': images, 'label': label}

In [ ]:
# Test it
s = r'![](images/mnist_ring_op_k_filmr_family.png) ![MNIST ring op$^k$ compositionality - FILMR \& Matrix Op](images/mnist_ring_op_k_mat_family.png)\{#fig:op_k}'
im_dict = parse_figure(s)
im_dict 

{'caption': 'MNIST ring op$^k$ compositionality - FILMR \\& Matrix Op',
 'images': [{'path': 'images/mnist_ring_op_k_filmr_family.png'},
  {'path': 'images/mnist_ring_op_k_mat_family.png'}],
 'label': 'op_k'}

In [ ]:
#| export
def make_figure(fig_dict: dict):
    "Generate LaTeX figure environment from image specs."
    images, caption, label = fig_dict['images'],  fig_dict.get('caption', ''), fig_dict.get('label', '')
    lines = ['\\begin{figure}[htbp]', '\\centering']  # Start figure environment
    for img in images:
        width_opt = f'[width={img["width"]}]' if 'width' in img else '[width=\\linewidth]' # Add width if specified, defalt to linewidth
        lines.append(f'\\includegraphics{width_opt}{{{img["path"]}}}')  # Include the image
    if caption: lines.append(f'\\caption{{{caption}}}')  # Add caption if provided
    if label: lines.append(f'\\label{{fig:{label}}}')  # Add label if provided
    lines.append('\\end{figure}')  # Close figure environment
    return '\n'.join(lines)

In [ ]:
fig_tex = make_figure(im_dict)
print(fig_tex)

\begin{figure}[htbp]
\centering
\includegraphics[width=\linewidth]{images/mnist_ring_op_k_filmr_family.png}
\includegraphics[width=\linewidth]{images/mnist_ring_op_k_mat_family.png}
\caption{MNIST ring op$^k$ compositionality - FILMR \& Matrix Op}
\label{fig:op_k}
\end{figure}


### Tables

In [ ]:
#| export
def parse_table(lines):
    """Parse markdown table with optional caption. Returns table_dict or None."""
    if isinstance(lines, str): lines = lines.split('\n')
    start = 0
    while start < len(lines) and not lines[start].strip(): start += 1
    lines = lines[start:]
    if not lines or not lines[0].startswith('|'): return None
    i = 0
    while i < len(lines) and lines[i].startswith('|'): i += 1
    if i < 3: return None
    sep = [c.strip() for c in lines[1].split('|')[1:-1]]
    aligns = ['c' if c.startswith(':') and c.endswith(':') else 'r' if c.endswith(':') else 'l' for c in sep]
    headers = [c.strip() for c in lines[0].split('|')[1:-1]]
    rows = [[c.strip() for c in r.split('|')[1:-1]] for r in lines[2:i]]
    caption, label = '', None
    if i < len(lines):
        m = re.match(r'\s*\*([^*]+)\*(?:\s*\\\{#([^}]+)\})?', lines[i])
        if m: caption, label = m.group(1), m.group(2)
    return {'headers': headers, 'rows': rows, 'alignments': aligns, 'caption': caption, 'label': label}

In [ ]:
# test it
test_str = """
| lambda-neg | TEST MRR ↑ | H@1 ↑ | H@3 ↑ | H@10 ↑ | MR(cos) ↓ |
|---|---|---|---|---|---|
| 0 | 0.457 | 0.395 | 0.487 | 0.573 | ~1300 |
| 0.003 | 0.4645 | 0.402 | 0.497 | 0.576 | 1236 |
| 0.01 | 0.4746 | 0.414 | 0.504 | 0.585 | 1233 |
| 0.03 | 0.489 | 0.431 | 0.519 | 0.592 | **1180** |
| 0.20 (champion) | **0.5151** | **0.4665** | **0.5327** | **0.6151** | 1749 |
 *The contrastive lever — `--lambda-neg` sweep (WN18RR, order2 nd512 lambd0.10)* \{#tab:contr}
"""
table_dict = parse_table(test_str) 
table_dict

<>:10: SyntaxWarning: invalid escape sequence '\{'


<>:10: SyntaxWarning: invalid escape sequence '\{'


/tmp/ipymini_3931/575591256.py:10: SyntaxWarning: invalid escape sequence '\{'
  *The contrastive lever — `--lambda-neg` sweep (WN18RR, order2 nd512 lambd0.10)* \{#tab:contr}


{'headers': ['lambda-neg',
  'TEST MRR ↑',
  'H@1 ↑',
  'H@3 ↑',
  'H@10 ↑',
  'MR(cos) ↓'],
 'rows': [['0', '0.457', '0.395', '0.487', '0.573', '~1300'],
  ['0.003', '0.4645', '0.402', '0.497', '0.576', '1236'],
  ['0.01', '0.4746', '0.414', '0.504', '0.585', '1233'],
  ['0.03', '0.489', '0.431', '0.519', '0.592', '**1180**'],
  ['0.20 (champion)',
   '**0.5151**',
   '**0.4665**',
   '**0.5327**',
   '**0.6151**',
   '1749']],
 'alignments': ['l', 'l', 'l', 'l', 'l', 'l'],
 'caption': 'The contrastive lever — `--lambda-neg` sweep (WN18RR, order2 nd512 lambd0.10)',
 'label': 'tab:contr'}

In [ ]:
#| export
def md_to_latex_bold(text: str):
    return re.sub(r'\*\*([^*]+)\*\*', r'\\textbf{\1}', text)

In [ ]:
md_to_latex_bold('| 0.20 (champion) | **0.5151** | **0.4665** | **0.5327** | **0.6151** | 1749 |')

'| 0.20 (champion) | \\textbf{0.5151} | \\textbf{0.4665} | \\textbf{0.5327} | \\textbf{0.6151} | 1749 |'

In [ ]:
test_str = r"""**Joint-Embedding Predictive Architectures.**
LeJEPA~\cite{balestriero2025lejepa} provides a theoretically grounded self-supervised objective combining an attraction loss with Sketched Isotropic Gaussian Regularization (SIGReg)."""
md_to_latex_bold(test_str)

'\\textbf{Joint-Embedding Predictive Architectures.}\nLeJEPA~\\cite{balestriero2025lejepa} provides a theoretically grounded self-supervised objective combining an attraction loss with Sketched Isotropic Gaussian Regularization (SIGReg).'

That just looks like the code I already had before that was printing those warning messages. .. or? Did you change anything?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're right, the function body is the same as what you originally had. The only change is removing the `warnings.disable(SyntaxWarning)` line that was causing the `AttributeError` — that was a failed attempt to suppress the warnings at runtime, which doesn't work.

The actual source of the `SyntaxWarning` is the *test string* itself:

```python
test_str = """...
LeJEPA~\cite{balestriero2025lejepa}..."""
```

`\c` and `\{` are invalid escape sequences in a regular string, so Python warns at parse time. Making it a raw string (`r"""..."""`) fixes that. The function never needed changing — it was already correct.

Well your change is to remove Change I added to try to get rid of the syntax warning. So you just reverted it to the original version.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, exactly. The function was already correct in its original form — the `warnings.disable` line was the only addition, and it was both wrong (no such method) and wouldn't have worked anyway.

The SyntaxWarning will still appear whenever you pass a non-raw string containing backslash sequences like `\c` or `\{`. The fix is purely on the calling side — use `r"""..."""` for those test strings.

In [ ]:
#| export
def make_table(tbl: dict):
    "Generate LaTeX table environment from parsed table dict."
    col_spec = ''.join(tbl['alignments'])
    lines = [f'\\begin{{table}}[htbp]', '\\centering', f'\\begin{{tabular}}{{{col_spec}}}', '\\hline']
    lines.append(' & '.join(tbl['headers']) + ' \\\\')
    lines.append('\\hline')
    for row in tbl['rows']:
        lines.append(' & '.join(md_to_latex_bold(cell) for cell in row) + ' \\\\')
    lines.extend(['\\hline', '\\end{tabular}'])
    if tbl.get('caption'): lines.append(f'\\caption{{{tbl["caption"]}}}')
    if tbl.get('label'): lines.append(f'\\label{{tab:{tbl["label"]}}}')
    lines.append('\\end{table}')
    return '\n'.join(lines)

In [ ]:
table_tex = make_table(table_dict)
print(table_tex)

\begin{table}[htbp]
\centering
\begin{tabular}{llllll}
\hline
lambda-neg & TEST MRR ↑ & H@1 ↑ & H@3 ↑ & H@10 ↑ & MR(cos) ↓ \\
\hline
0 & 0.457 & 0.395 & 0.487 & 0.573 & ~1300 \\
0.003 & 0.4645 & 0.402 & 0.497 & 0.576 & 1236 \\
0.01 & 0.4746 & 0.414 & 0.504 & 0.585 & 1233 \\
0.03 & 0.489 & 0.431 & 0.519 & 0.592 & \textbf{1180} \\
0.20 (champion) & \textbf{0.5151} & \textbf{0.4665} & \textbf{0.5327} & \textbf{0.6151} & 1749 \\
\hline
\end{tabular}
\caption{The contrastive lever — `--lambda-neg` sweep (WN18RR, order2 nd512 lambd0.10)}
\label{tab:tab:contr}
\end{table}


## Export Preserving Ordering via '#| replaces \<msgid\>' 

In [ ]:
#| export
def export_ordered(curr_path, output_path=None):
    """Uses user-defined syntax '#| replaces: <msg_id>' to replace earlier messages with later ones 
    msg_id be obtained in the GUI by pressing the link button.
    Valid syntax usages (with or without colons): 
        #| replaces: https://serene-vision-dives-ildq3w.solve.it.com/dialog_?name=solveit-tex/solveit-tex#_a0d44aac
        #| replaces _a0d44aac
    """
    if output_path is None: output_path = curr_path.replace('.ipynb', '-out.ipynb')
    nb = json.loads(Path(curr_path).read_text())
    
    # Pass 1: find replacements {target_id -> cell}, last-wins
    replacements = {}
    for cell in nb['cells']:
        content = ''.join(cell['source'])
        if not content.lstrip().startswith('#| export'): continue
        for line in content.split('\n'):
            if line.startswith('#| replaces:'):
                target = line.split('#')[-1].strip('_')
                replacements[target] = cell
    
    # Pass 2: walk in order, apply replacements, keep only exported
    out_cells = []
    for cell in nb['cells']:
        content = ''.join(cell['source'])
        if not content.lstrip().startswith('#| export'): continue
        cid = cell.get('id', '').strip('_')
        if cid in replacements:
            rcell = replacements[cid]
            rcontent = '\n'.join(l for l in ''.join(rcell['source']).split('\n') if not l.startswith('#| replaces'))
            lines = rcontent.split('\n')
            rcontent = [l + '\n' for l in lines[:-1]] + [lines[-1]]
            out_cells.append({**rcell, 'source': rcontent})
        else:
            if any(l.startswith('#| replaces') for l in content.split('\n')): continue  # don't re-add replacements
            out_cells.append(cell)
    
    nb['cells'] = out_cells
    Path(output_path).write_text(json.dumps(nb, indent=1))
    print(f'Exported {len(out_cells)} cells to {output_path}')
    return output_path 

In [ ]:
export_ordered(curr_dialog_path)

Exported 14 cells to /app/data/solveit-tex/solveit-tex-out.ipynb


'/app/data/solveit-tex/solveit-tex-out.ipynb'

## Conversion script for .ipynb -> .tex

In [ ]:
#| export
def export_ipynb_to_tex(ipynb_path: str, output_path: str = None, ordered=True):
    r"""Export a Solveit dialog (.ipynb) to a compilable LaTeX file.
    Cells are emitted in document order, each preceded by a `% <cell-id>` comment.
    The `## Abstract` cell emits `\begin{document}`, dividing preamble from document body."""

    ipynb_path = os.path.expanduser(ipynb_path)

    if ordered:  # Export preserving "#| replaces" ordering. Writes to -out.ipynb first. That becomes the input file 
        ipynb_path = export_ordered(ipynb_path)
        
    output_path = os.path.expanduser(output_path) if output_path else Path(ipynb_path).with_suffix('.tex')
    if ordered: output_path = str(output_path).replace('-out.tex', '.tex')
    print("output_path =",output_path)

    nb = json.loads(Path(ipynb_path).read_text())
    out = []

    for cell in nb['cells']:
        content = ''.join(cell['source'])

        if '#| export' not in content:
            continue

        filtered = '\n'.join(l for l in content.split('\n') if not l.startswith('#| '))
        out.append(f'% {cell["id"]}')

        if cell['cell_type'] == 'raw':
            out.append(filtered)
            continue

        lines = filtered.split('\n')
        i = 0
        while i < len(lines):
            line = lines[i]

            if line.startswith('# ') and not line.startswith('## '):
                out.append(f'\\title{{{line[2:].strip()}}}\n')
            elif line.startswith('\\author{'):
                out.append(line)
                while i < len(lines) and not lines[i].strip().endswith('}'):
                    i += 1
                    if i < len(lines):
                        out.append(lines[i])
            elif line == '## Abstract':
                out.append('\\begin{document}\n\n\\maketitle\n')
                out.append('\\begin{abstract}\n')
                i += 1
                while i < len(lines) and not lines[i].startswith('## '):
                    out.append(lines[i])
                    i += 1
                out.append('\\end{abstract}\n')
                continue
            elif line == '## References':
                out.append('\\small\n')
                i += 1
                while i < len(lines) and not lines[i].startswith('## '):
                    l = lines[i]
                    if l.strip().startswith('\\bibliographystyle'):
                        out.append(l + '\n')
                    bib_match = re.search(r'(\w+)\.bib', l)
                    if bib_match:
                        out.append(f'\\bibliography{{{bib_match.group(1)}}}\n')
                    i += 1
                continue
            elif line.startswith('### '):
                out.append(f'\\subsection{{{line[4:].strip()}}}\n')
            elif line.startswith('## '):
                out.append(f'\\section{{{line[3:].strip()}}}\n')
            elif line.startswith('|'):    # table handling
                # Collect table lines
                table_lines = []
                while i < len(lines) and lines[i].startswith('|'):
                    table_lines.append(lines[i])
                    i += 1
                # Check for caption line
                if i < len(lines) and re.match(r'\s*\*', lines[i]):
                    table_lines.append(lines[i])
                    i += 1
                # Parse and convert
                tbl_dict = parse_table(table_lines)
                if tbl_dict:
                    out.append(make_table(tbl_dict))
                continue  # Skip the i += 1 at the end    
            else:         # figures
                fig_dict = parse_figure(line)
                if fig_dict:
                    out.append(make_figure(fig_dict))
                else:
                    out.append(line)

            i += 1

    final = '\\documentclass{article}\n\\usepackage{graphicx}\n\n'
    final += '\n'.join(out) + '\n\n'
    final += '\\end{document}\n'
    Path(output_path).write_text(final)
    print(f'Created {output_path}')
    output_url = get_private_url(output_path)
    display(HTML(f'<a href="{output_url}" target="_blank">{output_url}</a>'))

In [ ]:
export_ipynb_to_tex('example/neurips_2026.ipynb')

Exported 22 cells to example/neurips_2026-out.ipynb


output_path = example/neurips_2026.tex


Created example/neurips_2026.tex


HTML(<a href="https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.tex" target="_blank">https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.tex</a>)

In [ ]:
export_ipynb_to_tex('~/hoplas-paper/hoplas-paper.ipynb')

Exported 13 cells to /app/data/hoplas-paper/hoplas-paper-out.ipynb


output_path = /app/data/hoplas-paper/hoplas-paper.tex


Created /app/data/hoplas-paper/hoplas-paper.tex


HTML(<a href="https://serene-vision-dives-ildq3w.solve.it.com/static/hoplas-paper/hoplas-paper.tex" target="_blank">https://serene-vision-dives-ildq3w.solve.it.com/static/hoplas-paper/hoplas-paper.tex</a>)

## Conversion script for tex -> pdf

In [ ]:
#| export
def compile_latex(tex_file: str, cwd: str = '.'):
    "Run full LaTeX compilation: pdflatex → bibtex → pdflatex → pdflatex"

    cwd = os.path.expanduser(cwd)
    tex_file = os.path.expanduser(tex_file)
    
    # Make path absolute before splitting
    full_tex_path = os.path.abspath(os.path.join(cwd, tex_file))
    work_dir = os.path.dirname(full_tex_path)
    
    base_name = os.path.splitext(os.path.basename(full_tex_path))[0]
    tex_content = Path(full_tex_path).read_text()
    bib_match = re.search(r'\\bibliography\{([^}]+)\}', tex_content)
    
    print("Running pdflatex (pass 1)...")
    r1 = subprocess.run(f'pdflatex -halt-on-error {full_tex_path}',
                   shell=True, cwd=work_dir, capture_output=True, text=True)
    if r1.returncode != 0:
        print(f"  FAILED:\n{r1.stdout[-3000:]}")
        return
    
    if bib_match:
        bib_name = bib_match.group(1)
        print(f"\nRunning bibtex ({bib_name})...")
        r = subprocess.run(f'bibtex {base_name}',
                          shell=True, cwd=work_dir, capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  FAILED:\n{r.stdout}")
            return
    
    for i in [2, 3]:
        print(f"\nRunning pdflatex (pass {i})...")
        r = subprocess.run(f'pdflatex -halt-on-error {full_tex_path}',
                       shell=True, cwd=work_dir, capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  FAILED:\n{r.stdout[-3000:]}")
            return

    pdf_path = full_tex_path.replace('.tex', '.pdf') 
    pdf_url = get_private_url(pdf_path)
    print(f"\nSuccess!")
    print(f"File: {pdf_path}")
    print(f"PDF url: {pdf_url}")
    sys.stdout.flush() 
    display(HTML(f'<a href="{pdf_url}" target="_blank">{pdf_url}</a>'))

In [ ]:
compile_latex('example/neurips_2026.tex')

Running pdflatex (pass 1)...



Running bibtex (references)...



Running pdflatex (pass 2)...



Running pdflatex (pass 3)...



Success!


File: /app/data/solveit-tex/example/neurips_2026.pdf


PDF url: https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.pdf


HTML(<a href="https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.pdf" target="_blank">https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/neurips_2026.pdf</a>)

In [ ]:
compile_latex('~/hoplas-paper/hoplas-paper.tex')

Running pdflatex (pass 1)...



Running bibtex (references)...


  FAILED:
This is BibTeX, Version 0.99e (TeX Live 2026)
The top-level auxiliary file: hoplas-paper.aux
I found no \bibstyle command---while reading file hoplas-paper.aux
(There was 1 error message)


## All-in-One Wrapper Function

In [ ]:
#| export
async def current_to_pdf():
    """
    Wrapper that converts the current dialogue to PDF and prints the private URL for it.
    Usage: await current_to_pdf()
    """
    curr_dialog_path = await get_curr_dialog_path()
    export_ipynb_to_tex(curr_dialog_path)
    display(HTML(f'<br>'))
    compile_latex(curr_dialog_path.replace('.ipynb', '.tex'))

## Submission Package (for ArXiv) 

In [ ]:
#| export
def create_submission_package(project_dir:str='.'): 
    "Make a .tar.gz archive suitable for submission to arxiv, etc"
    import shutil, tarfile
    from rgapi.skill import fd
    
    extensions = ['tex','png','jpg','tikz','eps','sty','bib','bst']
    project_path = Path(project_dir).expanduser().resolve()
    project_name = project_path.name
    tmp_dir = Path(f'/tmp/{project_name}')
    if tmp_dir.exists(): shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True)
    files = [Path(project_path) / f for f in fd(project_path, ext=extensions)]
    print("Files included:\n",[str(f) for f in files])
    for f in files:  
        dest = tmp_dir / f.relative_to(project_path)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest)
    pkg_path = Path(project_path) / f'{project_name}.tar.gz'
    with tarfile.open(pkg_path, 'w:gz') as tar:
        tar.add(tmp_dir, arcname=project_name)
    pkg_url = get_private_url(str(pkg_path))
    display(HTML(f'<a href="{pkg_url}" target="_blank">{pkg_url}</a>'))
    return str(pkg_path)

In [ ]:
create_submission_package('example')

Files included:


 ['/app/data/solveit-tex/example/neurips_2026.sty', '/app/data/solveit-tex/example/references.bib']


HTML(<a href="https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/example.tar.gz" target="_blank">https://serene-vision-dives-ildq3w.solve.it.com/static/solveit-tex/example/example.tar.gz</a>)

'/app/data/solveit-tex/example/example.tar.gz'

## Allow LLM to call routines herein

In [ ]:
#| export
from pyskills import allow 

allow(export_ipynb_to_tex)
allow(current_to_pdf)
allow(compile_latex) 
allow(create_submission_package)

<function __main__.create_submission_package(project_dir: str = '.')>